In [47]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [48]:
import json
import pandas as pd
from pathlib import Path


In [49]:
ROLE_PROFILE_PATH = "skill_gap/role_skill_profiles.csv"
role_profiles = pd.read_csv(ROLE_PROFILE_PATH)

# Canonical role list from data
VALID_ROLES = set(role_profiles["role_id"].unique())

sorted(VALID_ROLES)


['AI_ML_ENGINEER_INT',
 'DATA_ANALYST_INT',
 'DATA_ENGINEER_INT',
 'DEVOPS_TRAINEE',
 'INTERN_SE',
 'JR_BE_DEV',
 'JR_BUSINESS_ANALYST',
 'JR_FE_DEV',
 'JR_FS_DEV',
 'JR_IT_SUPPORT',
 'JR_MOBILE_DEV',
 'JR_QA_ENG',
 'JR_SE',
 'JR_SYS_ADMIN',
 'JR_UI_UX_DESIGNER']

In [50]:
CAREER_LADDER_PATH = "career_path/career_ladders.json"

with open(CAREER_LADDER_PATH, "r") as f:
    RAW_CAREER_LADDERS = json.load(f)

RAW_CAREER_LADDERS


{'SOFTWARE_ENGINEERING': ['INTERN_SE', 'JR_SE', 'JR_FE_DEV', 'JR_FS_DEV'],
 'DATA': ['DATA_ANALYST_INT', 'DATA_ENGINEER_INT'],
 'AI_ML': ['AI_ML_ENGINEER_INT'],
 'DEVOPS': ['DEVOPS_TRAINEE', 'JR_SYS_ADMIN'],
 'QA': ['JR_QA_ENG'],
 'MOBILE': ['JR_MOBILE_DEV'],
 'UI_UX': ['JR_UI_UX_DESIGNER']}

In [51]:
def detect_skill_gap(
    user_skill_ids: set,
    target_role_id: str,
    importance_threshold: float = 0.02
):
    role_df = role_profiles[role_profiles["role_id"] == target_role_id]

    if role_df.empty:
        raise ValueError(f"No role profile found for role_id={target_role_id}")

    required_skills = set(
        role_df[role_df["importance"] >= importance_threshold]["skill_id"]
    )

    missing_skills = sorted(required_skills - user_skill_ids)
    matched_skills = sorted(required_skills & user_skill_ids)

    readiness = (
        len(matched_skills) / len(required_skills)
        if required_skills else 0.0
    )

    return {
        "target_role": target_role_id,
        "readiness_score": round(readiness, 3),
        "missing_skills": missing_skills,
        "matched_skills": matched_skills
    }


In [52]:
def get_next_role(domain: str, current_role: str):
    ladder = CAREER_LADDERS.get(domain)

    if not ladder:
        raise ValueError(f"Unknown career domain: {domain}")

    if current_role not in ladder:
        raise ValueError(f"{current_role} not found in career ladder for {domain}")

    idx = ladder.index(current_role)

    if idx + 1 >= len(ladder):
        return None  # already at highest role

    return ladder[idx + 1]


In [53]:
# valid_roles = set(role_profiles["role_id"].unique())
# valid_roles


In [54]:
# def filter_ladders_to_valid_roles(ladders, valid_roles):
#     filtered = {}
#     for domain, roles in ladders.items():
#         filtered_roles = [r for r in roles if r in valid_roles]
#         if len(filtered_roles) >= 1:
#             filtered[domain] = filtered_roles
#     return filtered

# CAREER_LADDERS = filter_ladders_to_valid_roles(
#     CAREER_LADDERS,
#     valid_roles
# )

# CAREER_LADDERS


In [55]:
def simulate_career_path(
    domain: str,
    current_role: str,
    user_skill_ids: set,
    importance_threshold: float = 0.02
):
    """
    Integrated career pathway simulation with skill gap detection.
    - Uses dataset-backed career ladders
    - Fails gracefully if data is insufficient
    """

    # Step 1: determine next role
    try:
        next_role = get_next_role(domain, current_role)
    except ValueError as e:
        return {
            "current_role": current_role,
            "error": str(e)
        }

    if next_role is None:
        return {
            "domain": domain,
            "current_role": current_role,
            "message": "You are already at the highest role supported by current job market data."
        }

    # Step 2: run gap detection (graceful fallback)
    try:
        gap_result = detect_skill_gap(
            user_skill_ids=user_skill_ids,
            target_role_id=next_role,
            importance_threshold=importance_threshold
        )
    except ValueError:
        return {
            "domain": domain,
            "current_role": current_role,
            "next_role": next_role,
            "message": "Insufficient job data to evaluate skill gaps for this role."
        }

    # Step 3: return unified result
    return {
        "domain": domain,
        "current_role": current_role,
        "next_role": next_role,
        "readiness_score": gap_result["readiness_score"],
        "missing_skills": gap_result["missing_skills"],
        "matched_skills": gap_result["matched_skills"]
    }


In [56]:
user_skills = {
    "SK004",  # python
    "SK031",  # ml core
    "SK003"   # sql
}

simulate_career_path(
    domain="SOFTWARE_ENGINEERING",
    current_role="JR_SE",
    user_skill_ids={"SK003", "SK004", "SK007"}
)


{'domain': 'SOFTWARE_ENGINEERING',
 'current_role': 'JR_SE',
 'message': 'You are already at the highest role supported by current job market data.'}